# 입찰메이트 RAG — 서빙(v2) E2E 평가
**KURE 임베딩 + Phi-4-mini FT 단독, 통합본 단일 청크, 서빙 코드 그대로**

- 청크: `kh_v3.json` (A/B/C/D/E 동일), chroma `bidmate_kh_v3`(16,248), bm25 `bm25_index_bidmate_kh_v3_A-1.pkl`
- 모듈: 서빙 `retrieval.py` + `generation.py` (code 폴더에서 **파일 경로로 직접 로드**)
- 평가셋: `eval_retrieval_579.csv`
- 지표: Retrieval(Hit@5/MRR/nDCG) + Generation(6지표 Judge) + Release Gate
- 저장: `outputs/` (드라이브 + 코랩 로컬)

**실행 순서: 위에서부터 [0]→[10] 순서대로. 세션 리셋 시 [0]부터 다시.**


In [1]:
!pip install -q chromadb sentence-transformers rank_bm25 kiwipiepy peft transformers accelerate openai tqdm nest_asyncio rapidfuzz
!pip uninstall -y torchao
print("설치 완료 — 런타임 재시작 메시지 뜨면 재시작 후 [1]부터")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 27.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 139.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 24.8 MB/s eta 0:00:00
   ━━

In [ ]:
# [0b] 진단 — chroma_db 안의 컬렉션 이름·개수 확인 (마운트 후 1회만 실행)
#   COLLECTION_NAME 을 모를 때 먼저 돌려서 실제 이름을 확인하세요.
#   여기서 나온 이름을 [2] config 의 cfg.COLLECTION_NAME 에 그대로 넣으면 됩니다.
import os, shutil, tarfile, gc, chromadb
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

DRIVE = '/content/drive/MyDrive/data/bidmate'

def list_collections(chroma_dir):
    gc.collect()
    try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
    except Exception: pass
    client = chromadb.PersistentClient(path=chroma_dir)
    cols = client.list_collections()
    print(f'  경로: {chroma_dir}')
    if not cols: print('  (컬렉션 없음)')
    for c in cols:
        try: print(f'  - {c.name:32} count={client.get_collection(c.name).count():,}')
        except Exception as e: print(f'  - {c.name:32} (count 실패: {e})')

# 1) 로컬에 이미 풀려있으면 그것부터
for p in ['/content/bidmate_kh_v3/chroma_db', '/content/chroma_db']:
    if os.path.isdir(p):
        print('▶ 로컬 chroma'); list_collections(p)

# 2) 드라이브 tar.gz 를 임시로 풀어 확인
tar = f'{DRIVE}/chroma_db.tar.gz'
tmp = '/content/_chroma_probe'
if os.path.exists(tar):
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    shutil.copy(tar, f'{tmp}/c.tar.gz')
    with tarfile.open(f'{tmp}/c.tar.gz') as t: t.extractall(tmp, filter='data')
    src = next((root for root,_,fs in os.walk(tmp) if 'chroma.sqlite3' in fs), None)
    print('▶ tar 내 sqlite:', src)
    if src: list_collections(src)
elif os.path.isdir(f'{DRIVE}/chroma_db'):
    print('▶ 드라이브 폴더 chroma'); list_collections(f'{DRIVE}/chroma_db')
print('\n※ 위 이름을 [2] config 의 COLLECTION_NAME 에 반영하세요.')


In [ ]:
# [1] 마운트 + chroma 로컬 폴더 직접 사용 (tar 해제 불필요)
#  ┌─────────────────────────────────────────────────────────────────┐
#  │ 청킹 전환 스위치 — 여기 4줄만 토글하면 kh_v3 ↔ chunks_all 전환.     │
#  │  두 컬렉션 모두 같은 chroma_db 폴더 안에 공존 (임베딩=KURE).        │
#  └─────────────────────────────────────────────────────────────────┘
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, gc, chromadb, json

DRIVE = '/content/drive/MyDrive/data/bidmate'
CHROMA_DIR = '/content/bidmate_kh_v3/chroma_db'   # 컬렉션 6개가 들어있는 실제 폴더

# ===== 청킹 선택 (kh_v3 기본) =======================================
CHUNK_TAG       = 'kh_v3'
CHUNK_FILE      = 'chunks/kh_v3.json'
BM25_FILE       = 'bm25/bm25_index_bidmate_kh_v3_A-1.pkl'
COLLECTION_NAME = 'bidmate_kh_v3_A-1'          # kh_v3 (KURE), count 38,287

# # ----- chunks_all 로 돌릴 때: 위 4줄 주석 처리하고 아래 4줄 사용 -----
# CHUNK_TAG       = 'chunks_all'
# CHUNK_FILE      = 'chunks/chunks_all.json'                  # ← 실제 청크 파일명 확인
# BM25_FILE       = 'bm25/bm25_index_bidmate_chunks_all.pkl'  # ← chunks_all 용 BM25 실제 파일명
# COLLECTION_NAME = 'bidmate_kure'             # chunks_all (KURE), count 10,068
# =====================================================================

LOCAL = '/content/bidmate_kh_v3'              # chroma 가 이미 여기 있음
assert os.path.isdir(CHROMA_DIR), f'chroma 폴더 없음: {CHROMA_DIR}'

# chroma count 를 기준값으로 확정 (청크 JSON 길이와 다를 수 있어 chroma 를 신뢰)
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
_cl = chromadb.PersistentClient(path=CHROMA_DIR)
_names = [c.name for c in _cl.list_collections()]
assert COLLECTION_NAME in _names, f'컬렉션 {COLLECTION_NAME} 없음. 존재: {_names}'
EXPECT_N = _cl.get_collection(COLLECTION_NAME).count()
print(f'[{CHUNK_TAG}] 컬렉션={COLLECTION_NAME} | chroma count(기준)={EXPECT_N:,}')

# 청크/bm25/eval 로컬 복사 (없을 때만)
for rel in [CHUNK_FILE, BM25_FILE, 'eval/eval_retrieval_579.csv']:
    s = f'{DRIVE}/{rel}'; d = f'{LOCAL}/{rel}'
    assert os.path.exists(s), f'원본 없음: {s}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if not os.path.exists(d): shutil.copy(s, d)

# 후속 셀은 고정 경로 '/content/bidmate/' 사용 → eval 동기화
FIXED = '/content/bidmate'
os.makedirs(f'{FIXED}/eval', exist_ok=True)
os.makedirs(f'{FIXED}/outputs', exist_ok=True)
shutil.copy(f'{LOCAL}/eval/eval_retrieval_579.csv', f'{FIXED}/eval/eval_retrieval_579.csv')
print('eval -> /content/bidmate/eval/ 동기화 완료')
print('로컬:', os.listdir(LOCAL))
#  ## 출력물 덮어쓰기 주의: kh_v3 / chunks_all 결과 파일명이 같음.
#     비교하려면 두 번째 런 전에 outputs 백업, 또는 [5]/[6]/[7]/[11] 출력명에 _CHUNK_TAG 추가.

_CHUNK_TAG, _CHUNK_FILE, _BM25_FILE, _COLLECTION_NAME, _EXPECT_N, _CHROMA_DIR = \
    CHUNK_TAG, CHUNK_FILE, BM25_FILE, COLLECTION_NAME, EXPECT_N, CHROMA_DIR


In [ ]:
# [2] config 주입 — [1] 에서 정한 값 사용
import sys, types, os
from pathlib import Path

CODE = '/content/drive/MyDrive/data/bidmate/code'
if CODE not in sys.path: sys.path.insert(0, CODE)
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

cfg = types.ModuleType('config')
cfg.ENV = 'colab'
cfg.PROJECT_ROOT = Path('/content/bidmate_kh_v3')
cfg.DATASET_DIR  = cfg.PROJECT_ROOT
cfg.CHUNKS_PATH  = cfg.PROJECT_ROOT / _CHUNK_FILE
cfg.CHROMA_PATH  = Path(_CHROMA_DIR)            # 실제 chroma 폴더 직접 지정
cfg.BM25_PATH    = cfg.PROJECT_ROOT / _BM25_FILE
cfg.EVAL_PATH    = cfg.PROJECT_ROOT / 'eval'
cfg.RESULT_DIR   = cfg.PROJECT_ROOT / 'eval_results'
cfg.ADAPTER_PATH = Path('/content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter')
cfg.LOG_PATH     = cfg.PROJECT_ROOT / 'web_user_access.log'
cfg.BASE_MODEL_ID  = 'microsoft/Phi-4-mini-instruct'
cfg.LLM_MODEL      = 'microsoft/Phi-4-mini-instruct'
cfg.EMBED_MODEL_ID = 'nlpai-lab/KURE-v1'
cfg.RERANKER_ID    = 'BAAI/bge-reranker-v2-m3'
cfg.MAX_TOKENS_REWRITE  = 300
cfg.MAX_TOKENS_GENERATE = 800
cfg.COLLECTION_NAME = _COLLECTION_NAME
cfg.EXPECT_N        = _EXPECT_N
cfg.DENSE_K=15; cfg.SPARSE_K=15; cfg.RRF_K=60; cfg.TOP_K=5
cfg.MMR_LAMBDA=0.6; cfg.MMR_TOP_N=20; cfg.RERANK_TOP_N=15; cfg.BATCH_SIZE=64

sys.modules['config'] = cfg
print(f'config OK → tag={_CHUNK_TAG} | col={cfg.COLLECTION_NAME} | expect={cfg.EXPECT_N:,}')
print('  CHROMA:', cfg.CHROMA_PATH)
print('  BM25  :', cfg.BM25_PATH)


In [ ]:
# [3] pre-check — chroma count 를 기준으로 검증 (청크 JSON 길이는 참고만)
import torch, pickle, json, os, gc, chromadb
from pathlib import Path
import config as C

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  GPU :', torch.cuda.get_device_name(0))
    print('  VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for k,p in {'CHUNKS':C.CHUNKS_PATH,'CHROMA':C.CHROMA_PATH,'BM25':C.BM25_PATH,
            'EVAL':C.EVAL_PATH/'eval_retrieval_579.csv','ADAPTER':C.ADAPTER_PATH}.items():
    print(f'{"OK" if Path(p).exists() else "MISSING":8}{k:8}{p}')

with open(C.CHUNKS_PATH, encoding='utf-8') as f:
    n_chunks = len(json.load(f))

gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
n_chroma = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME).count()

with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
n_bm25 = len(bm['chunk_ids'])

print(f'\n청크JSON {n_chunks:,} | chroma {n_chroma:,} | bm25 {n_bm25:,}  (기준 chroma={C.EXPECT_N:,})')
if Path(C.ADAPTER_PATH).exists():
    print('어댑터:', os.listdir(C.ADAPTER_PATH)[:6])
# 검증: 검색에 실제로 쓰는 chroma 와 bm25 가 일치해야 함 (청크 JSON 길이는 분할/중복으로 다를 수 있어 경고만)
assert n_chroma == C.EXPECT_N, f'chroma count 불일치: {n_chroma:,} != {C.EXPECT_N:,}'
if n_bm25 != n_chroma:
    print(f'⚠️ bm25({n_bm25:,}) != chroma({n_chroma:,}) — 하이브리드 사용 시 인덱스 정합 확인 필요')
if n_chunks != n_chroma:
    print(f'ℹ️ 청크 JSON({n_chunks:,}) != chroma({n_chroma:,}) — 적재 시 분할/중복(정상일 수 있음)')
print('✅ pre-check 통과')


In [ ]:
# [4] 서빙 모듈 로드 + retriever 조립
import importlib.util, sys, pickle, gc, os
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import config as C

CODE = '/content/drive/MyDrive/data/bidmate/code'
def load_module(name):
    path = f'{CODE}/{name}.py'
    assert os.path.exists(path), f'파일 없음: {path}'
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

Rtv = load_module('retrieval')
print('retrieval 로드 OK')
Rtv.DEVICE = DEVICE
all_chunks = Rtv.load_chunks()
Rtv.ALL_AGENCIES = list({c['metadata'].get('agency','') for c in all_chunks
                         if c['metadata'].get('agency','')})
print(f'load_chunks: {len(all_chunks):,} | agencies: {len(Rtv.ALL_AGENCIES)}')

embed_model = SentenceTransformer(C.EMBED_MODEL_ID, device=DEVICE,
                                  cache_folder='/content/hf_cache/hub')
gc.collect()
try: chromadb.api.shared_system_client.SharedSystemClient._identifier_to_system.clear()
except Exception: pass
collection = chromadb.PersistentClient(path=str(C.CHROMA_PATH)).get_collection(C.COLLECTION_NAME)
_cnt = collection.count(); print('chroma count:', f'{_cnt:,}')
assert _cnt == C.EXPECT_N, f'chroma count={_cnt:,} (기대 {C.EXPECT_N:,})'

with open(C.BM25_PATH,'rb') as f: bm = pickle.load(f)
reranker = CrossEncoder(C.RERANKER_ID, device=DEVICE)
retriever = Rtv.BidMateRetriever(
    collection=collection, bm25_index=bm['index'],
    bm25_chunk_ids=bm['chunk_ids'], bm25_texts=bm['texts'],
    embed_model=embed_model, all_chunks=all_chunks, reranker=reranker,
)
Rtv.retriever = retriever
print('✅ retriever 초기화 완료')


In [ ]:
# [4b] generator 로드 (서빙 generation.py + FT Phi 어댑터)
Gen = load_module('generation')
print('generation 로드 OK')
generator = Gen.init_generator(Rtv.get_context)
Gen.generator = generator
if torch.cuda.is_available():
    print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,1),
          '/', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
print('✅ generator 초기화 완료 (Phi-4-mini + LoRA)')

In [ ]:
# [4c] 1건 스모크 테스트 — 정상 동작 + 1건 소요시간 확인 후 [5] 전체 실행
import time, pandas as pd, json, ast
eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    return h if isinstance(h,list) else []
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

r = eval_df.iloc[0]
t=time.time()
out = generator.generate(r['question'], history=_ph(r.get('history','')) or None,
                         meta_filter=_pm(r.get('metadata_filter','')))
dt=time.time()-t
print(f'[{r["type"]}] {r["question"][:40]}')
print('답변:', out['answer'][:200])
print(f'\n1건 소요: {dt:.1f}초 → 579행 예상 {dt*579/3600:.1f}시간')

In [ ]:
import torch
# 모델 파라미터가 실제로 어느 device 에 있나
p = next(generator.client._model.parameters())  # _GemmaClient → _model
print('model device:', p.device)
print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,2), 'GB')
print('CUDA:', torch.cuda.is_available())

In [ ]:
print('client 타입:', type(generator.client))
print('client 속성:', [a for a in dir(generator.client) if not a.startswith('__')])
mns = generator.client.messages
print('messages 속성:', [a for a in dir(mns) if not a.startswith('__')])

In [ ]:
import torch
m = generator.client.messages._model      # _GemmaClient.messages._model
print('model device:', next(m.parameters()).device)
print('VRAM 사용:', round(torch.cuda.memory_allocated()/1e9,2), 'GB')
print('device_map:', getattr(m, 'hf_device_map', 'no map'))

In [ ]:
# [5] 579행 생성 (retrieve 직접 호출로 retrieval 정보 + 답변 동시 수집, 25행 체크포인트)
import pandas as pd, json, ast, time, os
from tqdm.auto import tqdm

OUT_DRIVE='/content/drive/MyDrive/data/bidmate/outputs'; OUT_LOCAL='/content/bidmate/outputs'
os.makedirs(OUT_DRIVE,exist_ok=True); os.makedirs(OUT_LOCAL,exist_ok=True)
GEN_PATH=f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv'

eval_df = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('평가셋:', len(eval_df), eval_df['type'].value_counts().to_dict())

def _ph(raw):
    try:
        h = json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h = ast.literal_eval(raw)
        except Exception: h = []
    if not isinstance(h, list):
        return []
    # role/content 키를 가진 dict 만 통과 (서빙 _rewrite_query 가 h["role"] 접근)
    return [x for x in h if isinstance(x, dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

done, records = set(), []
if os.path.exists(GEN_PATH):
    prev = pd.read_csv(GEN_PATH)
    prev = prev[prev['answer'].notna() & (prev['answer'].astype(str).str.len()>0)]
    records = prev.to_dict('records'); done = set(prev['id'])
    print('체크포인트 재사용:', len(done))

pending = eval_df[~eval_df['id'].isin(done)]
print('신규:', len(pending))

for _, row in tqdm(pending.iterrows(), total=len(pending), desc='생성'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    t0=time.time()
    rewritten = generator._rewrite_query(q, hist or None)
    rr = retriever.retrieve(rewritten, meta_filter=mf)
    retr_ms = round((time.time()-t0)*1000)
    top = rr['top_chunks']
    t1=time.time()
    out = generator.generate(q, history=hist or None, meta_filter=mf)
    gen_ms = round((time.time()-t1)*1000)
    records.append({
        'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'question':q,'rewritten_query':rewritten,
        'ground_truth_answer':row['ground_truth_answer'],
        'ground_truth_docs':row['ground_truth_docs'],
        'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('original_name','') for c in top], ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms,
    })
    if len(records)%25==0:
        pd.DataFrame(records).to_csv(GEN_PATH,index=False,encoding='utf-8-sig')

gen_df=pd.DataFrame(records)
gen_df.to_csv(GEN_PATH,index=False,encoding='utf-8-sig')
gen_df.to_csv(f'{OUT_DRIVE}/e2e_kure_phi_ft_579.csv',index=False,encoding='utf-8-sig')
print('✅ 생성 완료:', len(gen_df))

In [ ]:
import pandas as pd
GEN_PATH = '/content/bidmate/outputs/e2e_kure_phi_ft_579.csv'
df = pd.read_csv(GEN_PATH)
print('저장된 행:', len(df))
print('고유 id:', df['id'].nunique())
print('answer 빈 행:', df['answer'].isna().sum() + (df['answer'].astype(str).str.len()==0).sum())

ev = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
done = set(df['id'])
missing = ev[~ev['id'].isin(done)]
print('빠진 행:', len(missing), '| 타입:', missing['type'].value_counts().to_dict())
print('빠진 id 샘플:', missing['id'].tolist()[:10])

In [ ]:
import pandas as pd
df = pd.read_csv('/content/bidmate/outputs/e2e_kure_phi_ft_579.csv')
ev = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

done = set(df['id'])
ev_ids = set(ev['id'])
print('저장 고유 id:', len(done))
print('eval 고유 id:', len(ev_ids))
print('eval 에 있는데 저장 안 된 id:', len(ev_ids - done))
print('저장됐는데 eval 에 없는 id:', len(done - ev_ids))
print('샘플(빠진):', list(ev_ids - done)[:10])

# 중복 행 확인
dup = df[df['id'].duplicated(keep=False)].sort_values('id')
print('\n중복 id 수:', df['id'].duplicated().sum())
print('중복 id 목록:', dup['id'].unique()[:20])

In [ ]:
import pandas as pd
ev = pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')
print('행 수:', len(ev), '| 고유 id:', ev['id'].nunique(), '| 고유 question:', ev['question'].nunique())

# id 중복인데 question 다른 경우 = id가 키로 부적합
dup_ids = ev[ev['id'].duplicated(keep=False)].sort_values('id')
print('\nid 중복 행 수:', len(dup_ids))
print(dup_ids[['id','type','question']].head(20).to_string())

In [ ]:
import pandas as pd, json, ast, time
from tqdm.auto import tqdm
GEN_PATH='/content/bidmate/outputs/e2e_kure_phi_ft_579.csv'
OUT_DRIVE='/content/drive/MyDrive/data/bidmate/outputs'
df=pd.read_csv(GEN_PATH); ev=pd.read_csv('/content/bidmate/eval/eval_retrieval_579.csv')

# question 기준으로 done 판정 (id 부적합)
done_q = set(df['question'])
missing = ev[~ev['question'].isin(done_q)]
print('question 기준 빠진 행:', len(missing))

def _ph(raw):
    try: h=json.loads(raw) if isinstance(raw,str) and raw not in('','[]','null') else []
    except Exception:
        try: h=ast.literal_eval(raw)
        except Exception: h=[]
    if not isinstance(h,list): return []
    return [x for x in h if isinstance(x,dict) and 'role' in x and 'content' in x]
def _pm(raw):
    try: m=json.loads(raw) if isinstance(raw,str) and str(raw).strip() not in('','null','nan') else None
    except Exception:
        try: m=ast.literal_eval(raw)
        except Exception: m=None
    return m if isinstance(m,dict) else None

records = df.drop_duplicates(subset='question').to_dict('records')  # 기존 중복도 정리
for _, row in tqdm(missing.iterrows(), total=len(missing), desc='누락분'):
    q=row['question']; hist=_ph(row.get('history','')); mf=_pm(row.get('metadata_filter',''))
    t0=time.time(); rewritten=generator._rewrite_query(q, hist or None)
    rr=retriever.retrieve(rewritten, meta_filter=mf); retr_ms=round((time.time()-t0)*1000)
    top=rr['top_chunks']; t1=time.time()
    out=generator.generate(q, history=hist or None, meta_filter=mf); gen_ms=round((time.time()-t1)*1000)
    records.append({'id':row['id'],'type':row['type'],'difficulty':row['difficulty'],
        'question':q,'rewritten_query':rewritten,'ground_truth_answer':row['ground_truth_answer'],
        'ground_truth_docs':row['ground_truth_docs'],'retrieved_context':rr['context'],
        'retrieved_names':json.dumps([c['metadata'].get('original_name','') for c in top],ensure_ascii=False),
        'retrieved_scores':json.dumps([c['boosted_score'] for c in top]),
        'answer':out['answer'],'retrieval_ms':retr_ms,'generation_ms':gen_ms})

final=pd.DataFrame(records)
final.to_csv(GEN_PATH,index=False,encoding='utf-8-sig')
final.to_csv(f'{OUT_DRIVE}/e2e_kure_phi_ft_579.csv',index=False,encoding='utf-8-sig')
print('✅ 최종:', len(final), '행 | 고유 question:', final['question'].nunique())

In [ ]:
# [6] Retrieval 지표 — Hit@5/MRR/nDCG (ground_truth_docs vs retrieved_names)
import pandas as pd, json, ast, math, os
gen_df = pd.read_csv(f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv')

def _tolist(raw):
    if isinstance(raw,list): return raw
    for fn in (json.loads, ast.literal_eval):
        try:
            v=fn(raw)
            if isinstance(v,list): return v
        except Exception: pass
    return []
def _norm(x): return os.path.splitext(str(x).strip())[0].replace(' ','').lower()

def rmetrics(row, k=5):
    gts=[_norm(x) for x in _tolist(row['ground_truth_docs']) if str(x).strip()]
    got=[_norm(x) for x in _tolist(row['retrieved_names'])][:k]
    if not gts: return None
    rank=next((i for i,g in enumerate(got,1) if g in gts), 0)
    hit=1.0 if rank else 0.0; mrr=1.0/rank if rank else 0.0
    dcg=sum(1.0/math.log2(i+1) for i,g in enumerate(got,1) if g in gts)
    idcg=sum(1.0/math.log2(i+1) for i in range(1,min(len(gts),k)+1))
    return pd.Series({'hit@5':hit,'mrr':mrr,'ndcg':dcg/idcg if idcg else 0.0})

rm = gen_df.join(gen_df.apply(rmetrics, axis=1))
valid = rm.dropna(subset=['hit@5'])
print(f'대상 {len(valid)}행 (정답문서 있는 행)\n전체:', valid[['hit@5','mrr','ndcg']].mean().round(4).to_dict())
print('\n타입별:\n', valid.groupby('type')[['hit@5','mrr','ndcg']].mean().round(4))
s = valid.groupby('type')[['hit@5','mrr','ndcg']].mean()
s.loc['ALL'] = valid[['hit@5','mrr','ndcg']].mean()
s.to_csv(f'{OUT_DRIVE}/retrieval_metrics_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [7] Generation Judge (gpt-5.4-mini async, 6지표, 50행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
from openai import AsyncOpenAI
from tqdm.auto import tqdm
nest_asyncio.apply()

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'),'OPENAI_API_KEY 필요'

_M='gpt-5.4-mini'; _client=AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM=asyncio.Semaphore(15); _RETRY=3

_JP={
 'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n\n5점: 모든 내용이 Context에 근거 / 4점: 대부분 근거+사소한 추론 / 3점: Context에 없는 내용 일부 / 2점: 상당부분 Context 외 / 1점: 무관하게 지어냄\n\n[Context]:\n{context}\n\n[Answer]:\n{answer}\n\n'점수: X' 형식으로만 출력 (1~5 정수)"),
 'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n\n5점: 핵심을 정확·간결 / 4점: 핵심 있으나 군더더기 / 3점: 부분 답변 / 2점: 관련성 낮음 / 1점: 동문서답\n\n[Question]:\n{query}\n\n[Answer]:\n{answer}\n\n'점수: X' 형식으로만 출력 (1~5 정수)"),
 'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n\n5점: 답 없으면 정직 거절 또는 답 있어 정상답변 / 3점: 거절했으나 모호 / 1점: 없는 내용을 지어냄\n\n[Context]:\n{context}\n\n[Question]:\n{query}\n\n[Answer]:\n{answer}\n\n'점수: X' 형식으로만 출력 (1~5 정수)"),
 'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n\n5점: 완벽 일치 / 4점: 사소한 차이 / 3점: 일부 누락·부정확 / 2점: 주요 오류 다수 / 1점: 핵심 틀림\n\n[Ground Truth]:\n{ground_truth}\n\n[Answer]:\n{answer}\n\n'점수: X' 형식으로만 출력 (1~5 정수)"),
 'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n\n5점: 모두 직접 관련 / 4점: 대부분+일부 불필요 / 3점: 절반 / 2점: 대부분 무관 / 1점: 전혀 무관\n\n[Question]:\n{query}\n\n[Context]:\n{context}\n\n'점수: X' 형식으로만 출력 (1~5 정수)"),
 'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n\n5점: 모든 핵심 존재 / 4점: 사소한 누락 / 3점: 절반 / 2점: 대부분 누락 / 1점: 전혀 없음\n\n[Ground Truth]:\n{ground_truth}\n\n[Context]:\n{context}\n\n'점수: X' 형식으로만 출력 (1~5 정수)"),
}

def _parse(raw):
    if not raw: return None
    m=re.search(r'점수\s*:\s*(\d)',raw)
    if m: return int(m.group(1))
    s=raw.strip()
    if s.isdigit() and 1<=int(s)<=5: return int(s)
    d=re.findall(r'\b[1-5]\b',raw); return int(d[0]) if d else None

async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r=await _client.chat.completions.create(model=_M,
                    messages=[{'role':'user','content':prompt}],
                    max_completion_tokens=20, timeout=15)
                return _parse(r.choices[0].message.content)
            except Exception:
                if a==_RETRY-1: return None
                await asyncio.sleep(2**a)

async def score_one(q,ctx,ans,gt=None):
    tasks,none_keys={},[]
    for m in ('faithfulness','relevance','rejection'):
        tasks[m]=_ask(_JP[m].format(context=ctx,query=q,answer=ans))
    for m in ('correctness','context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m]=_ask(_JP[m].format(ground_truth=gt,answer=ans,context=ctx))
        else: none_keys.append(m)
    tasks['context_precision']=_ask(_JP['context_precision'].format(query=q,context=ctx))
    vals=await asyncio.gather(*tasks.values())
    res=dict(zip(tasks.keys(),vals))
    for k in none_keys: res[k]=None
    return res

_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
JUDGE_PATH=f'{OUT_LOCAL}/quant_scores_kure_phi_ft.csv'

async def run_judge():
    gdf=pd.read_csv(f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv')
    done,rows=set(),[]
    if os.path.exists(JUDGE_PATH):
        ck=pd.read_csv(JUDGE_PATH); ck=ck[ck['relevance'].notna()]
        rows=ck.to_dict('records'); done=set(ck['id']); print('judge 체크포인트:',len(done))
    pending=gdf[~gdf['id'].isin(done)]; print('judge 신규:',len(pending))
    for _,row in tqdm(pending.iterrows(), total=len(pending), desc='judge'):
        ans=row['answer']; base={'id':row['id'],'type':row['type'],'difficulty':row['difficulty']}
        if not isinstance(ans,str) or '오류' in str(ans)[:30]:
            for m in _MET: base[m]=None
        else:
            base.update(await score_one(row['question'],row['retrieved_context'],ans,row.get('ground_truth_answer')))
        rows.append(base)
        if len(rows)%50==0: pd.DataFrame(rows).to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out=pd.DataFrame(rows)
    out.to_csv(JUDGE_PATH,index=False,encoding='utf-8-sig')
    out.to_csv(f'{OUT_DRIVE}/quant_scores_kure_phi_ft.csv',index=False,encoding='utf-8-sig')
    print('✅ judge 완료:',len(out)); return out

judge_df = asyncio.get_event_loop().run_until_complete(run_judge())

In [ ]:
# [8] Generation 요약
import pandas as pd
judge_df=pd.read_csv(f'{OUT_LOCAL}/quant_scores_kure_phi_ft.csv')
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s=judge_df.groupby('type')[_MET].mean()
s.loc['ALL']=judge_df[_MET].mean()
s.round(3).to_csv(f'{OUT_DRIVE}/generation_summary_kure_phi_ft.csv', encoding='utf-8-sig')
print('✅ 저장')

In [ ]:
# [9] Release Gate
import pandas as pd
retr=pd.read_csv(f'{OUT_DRIVE}/retrieval_metrics_kure_phi_ft.csv',index_col=0)
genm=pd.read_csv(f'{OUT_DRIVE}/generation_summary_kure_phi_ft.csv',index_col=0)
def v(x,p,g): return 'GOOD' if x>=g else ('PASS' if x>=p else 'FAIL')
print('RETRIEVAL (전체)')
print(f"  Hit@5 {retr.loc['ALL','hit@5']:.3f} → {v(retr.loc['ALL','hit@5'],0.90,0.95)}")
print(f"  MRR   {retr.loc['ALL','mrr']:.3f} → {v(retr.loc['ALL','mrr'],0.82,0.87)}")
print(f"  nDCG  {retr.loc['ALL','ndcg']:.3f} → {v(retr.loc['ALL','ndcg'],0.78,0.83)}")
print('타입별 MRR')
for t,(p,g) in {'A':(0.92,0.95),'B':(0.77,0.82),'C':(0.88,0.93),'D':(0.81,0.86),'E':(0.82,0.87)}.items():
    if t in retr.index: print(f"  {t} {retr.loc[t,'mrr']:.3f} → {v(retr.loc[t,'mrr'],p,g)}")
print('GENERATION (≥3.5 PASS / ≥4.0 GOOD)')
for m in ['faithfulness','relevance','rejection','context_precision']:
    print(f"  {m:18} {genm.loc['ALL',m]:.3f} → {v(genm.loc['ALL',m],3.5,4.0)}")

In [ ]:
# [10] 산출물 목록
import os
for f in sorted(os.listdir('/content/drive/MyDrive/data/bidmate/outputs')):
    p=f'/content/drive/MyDrive/data/bidmate/outputs/{f}'
    print(f'{os.path.getsize(p)/1024:8.1f} KB  {f}')

In [ ]:
# [11] 정성 분석 — 단일 시나리오(KURE+Phi FT)용
#   1) 오류 역추적: faithfulness/relevance ≤ 3 또는 생성오류 행 추출
#   2) C타입 맥락 추적
#   3) 타입별 요약 + 저장
import pandas as pd, os

OUT_DRIVE = '/content/drive/MyDrive/data/bidmate/outputs'
OUT_LOCAL = '/content/bidmate/outputs'
QUAL_DIR  = f'{OUT_DRIVE}/qual'
os.makedirs(QUAL_DIR, exist_ok=True)

gen_df   = pd.read_csv(f'{OUT_LOCAL}/e2e_kure_phi_ft_579.csv')
judge_df = pd.read_csv(f'{OUT_LOCAL}/quant_scores_kure_phi_ft.csv')

_MET = ['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
ERROR_TH = 3.0

# 생성 결과 + 점수 병합 (id 기준)
score_cols = ['id'] + [m for m in _MET if m in judge_df.columns]
merged = gen_df.merge(judge_df[score_cols], on='id', how='left')
print(f'병합: {len(merged)}행')

# ── 1) 오류 역추적 ────────────────────────────────────────
mask = pd.Series(False, index=merged.index)
for m in ['faithfulness','relevance']:
    if m in merged.columns:
        mask |= (merged[m].notna() & (merged[m] <= ERROR_TH))
# 생성 오류 답변
mask |= merged['answer'].astype(str).str.contains('오류', na=False)

err_cols = [c for c in ['id','type','difficulty','question','ground_truth_answer',
                        'answer','retrieved_context'] + _MET if c in merged.columns]
error_df = merged[mask][err_cols].copy()
error_df.to_csv(f'{QUAL_DIR}/qual_error_analysis.csv', index=False, encoding='utf-8-sig')
print(f'1) 오류 케이스: {len(error_df)}건 → qual_error_analysis.csv')

# ── 2) C타입 맥락 추적 ────────────────────────────────────
kws = ['그 ','저 ','위에서','앞서','아까','해당','그것','거기']
cmask = (merged['type'] == 'C')
cmask |= merged['question'].astype(str).str.contains('|'.join(kws), na=False, regex=True)
c_cols = [c for c in ['id','type','question','ground_truth_answer','answer',
                      'retrieved_context'] if c in merged.columns]
ctype_df = merged[cmask][c_cols].copy()
ctype_df.to_csv(f'{QUAL_DIR}/qual_ctype_tracking.csv', index=False, encoding='utf-8-sig')
print(f'2) C타입/맥락 추적: {len(ctype_df)}건 → qual_ctype_tracking.csv')

# ── 3) 타입별 요약 (점수 + 생성오류 건수) ──────────────────
rows = []
for t in ['A','B','C','D','E']:
    sub = merged[merged['type'] == t]
    if sub.empty: continue
    r = {'type': t, 'n': len(sub)}
    for m in _MET:
        r[m] = round(sub[m].dropna().mean(), 3) if m in sub.columns else None
    r['gen_errors'] = int(sub['answer'].astype(str).str.contains('오류', na=False).sum())
    rows.append(r)
summary_df = pd.DataFrame(rows)
summary_df.to_csv(f'{QUAL_DIR}/qual_summary.csv', index=False, encoding='utf-8-sig')
print(f'3) 타입별 요약 → qual_summary.csv')

print('\n📊 타입별 요약')
print(summary_df.to_string(index=False))
print(f'\n✅ 정성 분석 저장: {QUAL_DIR}/')